### Understanding the `umask` Command
[LabEx article](https://labex.io/tutorials/linux-linux-umask-command-with-practical-examples-422970)

It is used to **set the default permissions** for **newly created** files and directories.

The `umask` command specifies the **file mode creation mask**, which determines the **default permissions** for new files and directories. The `umask` value is a 4-digit octal number that represents the permissions that **should not be granted**.

Checking the current umask value

In [ ]:
dennie@HP:~/linux-tut$ umask
0022

The `umask` value `0022` means that the default permissions for new files will be `0644` (`rw-r--r--`) and the default permissions for new directories will be `0755` (`rwxr-xr-x`).

> The `umask` value is calculated by subtracting the specified value from the default permissions (`0777` for files and `0755` for directories). For example, with a `umask` of `0022`, the default permissions will be:

- Files: `0777 - 0022 = 0755` (`rw-r--r--`)
- Directories: `0755 - 0022 = 0733` (`rwxr-xr--`)

You can change the `umask` value using the following command:

In [ ]:
dennie@HP:~/linux-tut$ umask 0002

This sets the `umask` to `0002`, which means that the default permissions for new files will be `0775` (`rw-rwxr-x`) and the default permissions for new directories will be `0775` (`rwxrwxr-x`).

---

#### Modifying File and Directory Permissions with `umask`
Create a new file and directory to observe the effect of umask:

In [ ]:
dennie@HP:~/linux-tut$ touch new_file
.txt
dennie@HP:~/linux-tut$ mkdir new_dire
ctory

In [ ]:
dennie@HP:~/linux-tut$ ls -l
total 208
-rw-r--r-- 1 dennie dennie      0 Sep  4 18:32 new_file.txt  # 0644
drwxr-xr-x 2 dennie dennie   4096 Sep  4 18:33 new_directory  # 0755

As you can see, the default permissions for the new file are `0644` (`rw-r--r--`) and the default permissions for the new directory are `0755` (`rwxr-xr-x`), which corresponds to the current `umask` value of `0022`.

Change the umask value to `0002`:

In [ ]:
dennie@HP:~/linux-tut$ umask 0002

Let's create a new file and directory again:

In [ ]:
dennie@HP:~/linux-tut$ touch another_file.txt && mkdir another_directory

And check the permissions:

In [ ]:
dennie@HP:~/linux-tut$ ls -l
total 212
-rw-rw-r-- 1 dennie dennie      0 Sep  5 08:39 another_file.txt  # 0774
drwxrwxr-x 2 dennie dennie   4096 Sep  5 08:39 another_directory  # 0775

As you can see, the default permissions for the new file are `0774` (`rw-rw-r--`) and the default permissions for the new directory are `0775` (`rwxrwxr-x`), which corresponds to the umask value of `0002`.

This demonstrates how you can use the `umask` command to modify the default permissions for newly created files and directories.

---

### Applying umask in Different Scenarios
In this step, we will explore how to apply the `umask` command in different scenarios to manage file and directory permissions effectively.

#### Scenario 1: Securing Sensitive Files
Imagine you have a sensitive file that you want to ensure only the owner has read and write access to. You can achieve this by setting a more restrictive `umask` value:

In [ ]:
dennie@HP:~/linux-tut$ umask 0077

In [ ]:
mkdir labex && touch labex/sensitive_file.txt

In [ ]:
stat labex/sensitive_file.txt 
  File: labex/sensitive_file.txt
  Size: 0               Blocks: 0      IO Block: 4096   regular emptyfile
Device: 8,48    Inode: 51769       Links: 1
Access: (0600/-rw-------)  Uid: ( 1000/  dennie)   Gid: ( 1000/  dennie)
Access: 2026-09-05 08:46:29.632452147 +0000
Modify: 2026-09-05 08:46:29.632452147 +0000
Change: 2026-09-05 08:46:29.632452147 +0000
 Birth: 2026-09-05 08:46:29.632452147 +0000

For a newly created **regular file**, the base permission is typically `0666` (`rw-rw-rw-`). The `umask` removes permissions:

In [ ]:
  0666
- 0077
= 0600

So:

In [ ]:
umask 0077
mkdir labex && touch labex/sensitive_file.txt
ls -l labex/sensitive_file.txt

produces:

In [ ]:
-rw------- 1 labex labex 0 ... sensitive_file.txt

This means:

- **Owner**: read and write (`rw-`)
- **Group**: no permissions (`---`)
- **Others**: no permissions (`---`)

For a newly created **directory**, the base permission is typically `0777` (`rwxrwxrwx`):

In [ ]:
mkdir labex/sensitive_dir
stat labex/sensitive_dir/
  File: labex/sensitive_dir/
  Size: 4096            Blocks: 8      IO Block: 4096   directory
Device: 8,48    Inode: 51789       Links: 2
Access: (0700/drwx------)  Uid: ( 1000/  dennie)   Gid: ( 1000/  dennie)
Access: 2026-09-05 09:27:58.296831473 +0000
Modify: 2026-09-05 09:27:58.296831473 +0000
Change: 2026-09-05 09:27:58.296831473 +0000
 Birth: 2026-09-05 09:27:58.296831473 +0000

In [ ]:
  0777
- 0077
= 0700

Therefore, `umask 0077` creates a directory with:

In [ ]:
drwx------

The `umask` value of `0077` ensures that the new file has permissions of `0700` (`rw-------`), which means only the owner has read and write access.

#### Scenario 2: Allowing Group Collaboration
In a scenario where you want to create files and directories that are accessible to a specific group, you can set the umask value accordingly:

In [ ]:
dennie@HP:~/linux-tut$ umask 0007
dennie@HP:~/linux-tut$ mkdir labex/shared_directory && touch labex/shared_file.txt

In [ ]:
dennie@HP:~/linux-tut$ ls -l labex/shared_file.txt 
-rw-rw---- 1 dennie dennie 0 Sep  5 09:30 labex/shared_file.txt

In [ ]:
dennie@HP:~/linux-tut$ ls -ld labex/shared_directory/
drwxrwx--- 2 dennie dennie 4096 Sep5 09:30 labex/shared_directory/

The `umask` value of `0007` ensures that the new directory has permissions of `0770` (`rwxrwx---`) and the new file has permissions of `0660` (`rw-rw----`), allowing **members of the group** to access and modify the files and directories.

> Remember, the `umask` value is subtracted from the default permissions (`0777` for files and `0755` for directories) to determine the final permissions.